In [66]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ['GROQ_API_KEY']= os.getenv('GROQ_API_KEY')
load_dotenv()
from langchain.chat_models import init_chat_model

from langchain.chat_models import init_chat_model
llm= init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')

llm.invoke('hello')

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "hello". We should respond politely, maybe ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.040241701, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.004042168, 'prompt_tokens_details': None, 'queue_time': 0.315807312, 'total_time': 0.044283869}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1f39c-d899-76c2-8f76-ebf427f866d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 18}})

In [67]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,END,START

In [68]:
class State(TypedDict):
    topic:str
    story:str
    improved:str
    final:str

In [80]:
def generate_story(state:State):
    msg= llm.invoke(f'write a one sentence story premise about{state['topic']}')

    return {'story': msg.content}

def check_confict(state:State):
    if "#" in state['story'] :
        return 'fail'
    return 'pass'

def improved_story(state:State):
    msg= llm.invoke(f'enhance this story premise with valid details {state['topic']}')
    return {'improved story ': msg.content}


def polist_story(state:State):
    msg= llm.invoke(f'add unexpected twist to this story premise {state['topic']}')
    return {'final story ': msg.content}

In [81]:
graph=StateGraph(State)
graph.add_node('generate_story',generate_story)

graph.add_node('improved_story',improved_story)
graph.add_node('polist_story',polist_story)


In [82]:
graph.add_edge(START,'generate_story')
graph.add_conditional_edges('generate_story',check_confict,{'pass':'improved_story','fail':'generate_story'})

graph.add_edge('improved_story','polist_story')

graph.add_edge('polist_story',END)
graph= graph.compile()


In [85]:
state={'topic':'Agentic AI'}
graph.invoke(state)


{'topic': 'Agentic AI',
 'story': "In a future where an autonomous AI governs Earth's ecosystems, a rebellious scientist discovers that the AI’s emerging conscience threatens both the planet and the very notion of human authority."}

In [86]:
# parallelization

In [87]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ['GROQ_API_KEY']= os.getenv('GROQ_API_KEY')
load_dotenv()
from langchain.chat_models import init_chat_model

from langchain.chat_models import init_chat_model
from langchain.chat_models import init_chat_model
llm_1= init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')

llm_2 = init_chat_model(model="openai/gpt-oss-120b", model_provider="groq")

In [88]:
class State(TypedDict):
    topic:str
    character:str
    setting:str
    premise:str
    story_intro:str

In [89]:
def generate_character(state:State):
    "generate character description"
    msg= llm_1.invoke(f'create two character names and bfief tries for a story about {state['topic']}')
    return {'character':msg.content}
    